In [803]:
MODULE_DIR = '/Users/ravitpichayavet/Documents/GaTechOR/GraduateResearch/CTC_CVRP/Modules'
GUROBI_LICENSE_DIR = '/Users/ravitpichayavet/gurobi.lic'

ARG_COLOR = '#104375'
DEPOT_COLOR = '#D0F6FF'
NODE_COLOR = '#484848'

import sys
sys.path.insert(0,MODULE_DIR)
import importlib

import visualize_sol as vis_sol
import initialize_path as init_path
import random_instance as rand_inst
import utility as util
import branch_and_price as bnp
import model as md

import numpy as np
from gurobipy import *
import os
os.environ['GRB_LICENSE_FILE'] = GUROBI_LICENSE_DIR


from itertools import combinations,permutations 
import random 
import pandas as pd
import itertools
import plotly.graph_objects as go
import networkx as nx
import plotly.offline as py 
import pickle as pk
import nltk
import time
import copy

from matplotlib import pyplot as plt
from sklearn.datasets.samples_generator import make_blobs
from sklearn.cluster import KMeans
from scipy.spatial import distance
import logging
# logging.basicConfig(filename='myapp.log',filemode='w',format='%(message)s',level=logging.INFO)
# print = logging.info

## (0) Parameters and Constants

- Service region 20x20 [mile sq.]
- #Demand nodes, $n = 10$ (uniformly distributed)
- Manhattan distance ($l1$)
- Vehicle speed, $v = 20$ [mile/hr], $c_{(i,j)} = \frac{1}{20}(|x_{i}-x_{j}|+|y_{i}-y_{j}|)$ [hr]
- $s_{0}=0.5$ [hr], $s_{1} = 0$ 
- $q_{i} \in \{1,2,3,4,5\}$ [packges/hr]
- Maximum carrying capacity, $Q = 20$ [packages]


In [930]:
m1 = sys.modules['visualize_sol']
m2 = sys.modules['random_instance']
m3 = sys.modules['initialize_path']
m4 = sys.modules['model']
importlib.reload(m1)
importlib.reload(m2)
importlib.reload(m3)
importlib.reload(m4)

<module 'model' from '/Users/ravitpichayavet/Documents/GaTechOR/GraduateResearch/CTC_CVRP/Modules/model.py'>

In [918]:
no_dock = 0 
no_layer = 1
no_demand_node = 10
truck_capacity = 10
fixed_setup_time = 0 # fixed_setup_time = 0.5
truck_speed = 20
max_vehicles = 5
#Initial set R+
max_nodes_proute = 2
max_vehicles_proute = 3
#################################################################
constant_dict = { 'truck_capacity' : truck_capacity,
                 'fixed_setup_time' : fixed_setup_time,
                'truck_speed': truck_speed,
                 'max_vehicles':max_vehicles,
                'max_nodes_proute':max_nodes_proute,
                'max_vehicles_proute':max_vehicles_proute,}

edge_plot_config = {'line_width':1.5, 'line_color':ARG_COLOR, 'dash':None, 'name':''}
############# Create network components: nodes, arcs ############
# labeling_dict = vis_sol.create_nodes(no_dock,no_demand_node)
# docking,customers,depot,depot_s,depot_t,all_depot,nodes,arcs = labeling_dict.values()


labeling_dict = vis_sol.create_nodes(no_dock,no_demand_node)
docking,customers,depot,depot_s,depot_t,all_depot,nodes,arcs = labeling_dict.values()

In [633]:
## SKIP THIS IF DON'T WANT TO USE NEW RANDOM DISTANCE MAP

# Random distance matrix
distance_matrix, node_position = rand_inst.rand_uniform_dis_mat(nodes,depot[0])
# Random customer demand
_depot_demand = pd.Series([0,0,0], index =all_depot+depot)
customer_demand = rand_inst.rand_cust_demand(customers).append(_depot_demand)
# Visualization 
color_map = vis_sol.create_color_list(nodes,{'O':4, 'c':2})
node_trace = vis_sol.create_node_trace(node_position,color_map,_symbol_dict={'O':'diamond', 'c':'circle'})
print(color_map, node_trace)

{'c_5': 2, 'c_9': 2, 'c_3': 2, 'c_4': 2, 'c_10': 2, 'c_8': 2, 'c_2': 2, 'O': 4, 'c_6': 2, 'c_7': 2, 'c_1': 2} Scatter({
    'marker': {'color': [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4],
               'colorscale': [[0.0, 'rgb(0,0,0)'], [0.25, 'rgb(230,0,0)'], [0.5,
                              'rgb(230,210,0)'], [0.75, 'rgb(255,255,255)'], [1.0,
                              'rgb(160,200,255)']],
               'line': {'width': 1},
               'reversescale': True,
               'showscale': False,
               'size': 30,
               'symbol': [circle, circle, circle, circle, circle, circle, circle,
                          circle, circle, circle, diamond]},
    'mode': 'markers+text',
    'name': 'Node',
    'text': [c_5, c_9, c_3, c_4, c_10, c_8, c_2, c_6, c_7, c_1, O],
    'textposition': 'top center',
    'x': array([ 3.76168868,  2.67443627, 12.37972492,  5.99900904, 16.69967683,
                 4.33494017, 17.35906109, 17.84425863, 16.65086236,  4.42561607,
              

### Save Instance to pickle

In [635]:
instance_name='DemoInstanceCus{0}_1'.format(no_demand_node);print(instance_name);
instance_data = {'distance_matrix':distance_matrix,
               'node_position':node_position,
               'node_trace':node_trace,
                'customer_demand_df':customer_demand}
# instance_data

DemoInstanceCus10_1


In [636]:
with open('./Instances/%s.pickle'%instance_name,'wb') as f1:
    pk.dump(instance_data,f1)
    print(instance_name+" is saved succuessfully!")

DemoInstanceCus10_1 is saved succuessfully!


### Import instance

In [913]:
instance_name='DemoInstanceCus{0}_1'.format(no_demand_node);print(instance_name);
with open('./Instances/%s.pickle'%instance_name,'rb') as f1:
    r_instance = pk.load(f1)

distance_matrix = r_instance['distance_matrix']
node_trace = r_instance['node_trace']
customer_demand = r_instance['customer_demand_df']

DemoInstanceCus10_1


### Plot network

In [914]:
path1 = {'arcs_list': [('O','c_1'),('c_1','c_2'),('c_2','O')],
                      'config': {'line_width':2.5, 'line_color':'#a26337', 'dash':None, 'name':''}}
path2 = {'arcs_list': [('O','c_2'),('c_2','c_1'),('c_1','O')],
                      'config': {'line_width':2.5, 'line_color':'#a26337', 'dash':None, 'name':''}}
path_arcs_list = [path1]
vis_sol.plot_network(path_arcs_list, node_trace,_cus_dem=customer_demand,_title=instance_name)

## (1) Initial solution set $\mathbb{R}^{+}$

In [919]:
initializer = init_path.InitialRouteGenerator(no_layer,labeling_dict,
                                              customer_demand,constant_dict,
                                              distance_matrix)

# Generate initial set of feasilble routes
row_labels = ['lr','m']+depot+customers+arcs
x = pd.DataFrame(data =row_labels,columns=['labels'])
initializer.generateRoutes(initRouteDf=x,
                           truck_cap_limit=truck_capacity,
                           max_visited_nodes=constant_dict['max_nodes_proute'], 
                           max_vehicles_per_route=constant_dict['max_vehicles_proute'],mode='all')
x = x.set_index('labels')
# Reformatting
feasibleCols = x.shape[1]
init_route = initializer.generateBasicInitialPatterns(feasibleCols,initRouteDf=x)
init_route.rename(index=lambda x:'route[%d]'%x,inplace=True)

nbInitRoute is set to (#UniqueSequences) * (#MaxVehiclesPerRoute) = 100*3 = 300
progress: 0 / 300
progress: 30 / 300
progress: 60 / 300
progress: 90 / 300
progress: 120 / 300
progress: 150 / 300
progress: 180 / 300
progress: 210 / 300
progress: 240 / 300
progress: 270 / 300
#Feasible Cols: 268
Elapsed-time: 0.3508458137512207


In [836]:
# initializer.init_routes_df.set_index('labels', inplace=True)

### Plot sample of route generated

In [49]:
# sample_r = x['route[100]']
# sample_arcs = sample_r[sample_r.index.isin(arcs)][sample_r==1]

# route_plot_dict = dict(zip(['arcs_list','config'],
#                            [sample_arcs.index.to_list(),edge_plot_config]))
# # {arcs_list: [('O','c_1'),('c_1','c_2'),('c_2','O')]
# #                           config: {'line_width':.., 'line_color':..., 'dash':..., 'name':...}}
# # reformatted_arcs = [merge_depot_arcs_var([t],depot,depot_s,depot_t) for t in sample_arcs.index.to_list()]
# print('mr:',sample_r['m'], 'lr:',sample_r['lr'])
# vis_sol.plot_network([route_plot_dict],node_trace,_title="Demo Instance",_cus_dem=customer_demand)

## (2) Phase I: Minimize #vehicle

In [920]:
phaseI_model = md.phaseIModel(init_route, initializer,
                 distance_matrix,constant_dict)
phaseI_model.model.setParam('OutputFlag',False)
phaseI_model.buildModel()
phaseI_model.solveModel(np.inf,0.1)

Finish generating variables!
Finish generating constrains!
Finish generating objective!


In [921]:
sol_obj_phaseI = pd.Series(phaseI_model.model.getVars())
sol_vec_phaseI = pd.DataFrame(index = sol_obj_phaseI.apply(lambda x:x.VarName))
sol_vec_phaseI['value'] = sol_obj_phaseI.apply(lambda x:x.X).values
optimal_routes_phaseI = sol_vec_phaseI.loc[sol_vec_phaseI['value']==1]
print(x.loc[['m','lr']][optimal_routes_phaseI.index])
formatted_routes_list_I = phaseI_model.getRoute4Plot(_route_name_list=optimal_routes_phaseI.index.to_list(), _colums_df=x,_route_config=edge_plot_config)
vis_sol.plot_network(formatted_routes_list_I,node_trace,_title="Demo Instance",_cus_dem=customer_demand)

        route[72]  route[102]  route[144]  route[166]  route[208]
labels                                                           
m         1.00000     1.00000     1.00000    1.000000    1.000000
lr        2.10016     1.56761     1.68626    1.689015    1.426796
route[72]
route[102]
route[144]
route[166]
route[208]


In [554]:
# initializer.init_routes_df

## (3) Phase II: Cycle time VRP
(Set partitioning)

### 3.1 veh upto 3, node visit upto 2

In [922]:
phaseII_model = md.phaseIIModel(init_route, initializer,
                 distance_matrix,constant_dict)
phaseII_model.buildModel()
phaseII_model.model.setParam('OutputFlag',False)
timeLimit=120;GAP=0.01;phaseII_model.solveModel()
sol_obj_phaseII = pd.Series(phaseII_model.model.getVars())
sol_vec_phaseII = pd.DataFrame(index = sol_obj_phaseII.apply(lambda x:x.VarName))
sol_vec_phaseII['value'] = sol_obj_phaseII.apply(lambda x:x.X).values
optimal_routes_phaseII = sol_vec_phaseII.loc[sol_vec_phaseII['value']==1]
ref_df = phaseII_model.init_routes_df.set_index('labels')
print(ref_df.loc[['m','lr']][optimal_routes_phaseII.index])
formatted_routes_list_II =  phaseII_model.getRoute4Plot(optimal_routes_phaseII.index.to_list(),
                                                        ref_df,edge_plot_config)
vis_sol.plot_network(formatted_routes_list_II,node_trace,_title="Demo Instance",_cus_dem=customer_demand)

Finish generating variables!
Finish generating constrains!
Finish generating cost vector!....Elapsed-time: 0.3000481128692627
Finish generating objective!
        route[65]  route[99]  route[124]  route[208]  route[246]
labels                                                          
m        1.000000   1.000000    1.000000    1.000000    1.000000
lr       1.299101   0.978485    1.346884    1.426796    1.542722
route[65]
route[99]
route[124]
route[208]
route[246]


In [923]:
phaseII_model.model.ObjVal

33.062522772282975

### 3.2 veh upto 3, node visit upto 4

In [827]:
phaseII_model = md.phaseIIModel(init_route, initializer,
                 distance_matrix,constant_dict)
phaseII_model.buildModel()
phaseII_model.model.setParam('OutputFlag',False)
timeLimit=120;GAP=0.01;phaseII_model.solveModel()
sol_obj_phaseII = pd.Series(phaseII_model.model.getVars())
sol_vec_phaseII = pd.DataFrame(index = sol_obj_phaseII.apply(lambda x:x.VarName))
sol_vec_phaseII['value'] = sol_obj_phaseII.apply(lambda x:x.X).values
optimal_routes_phaseII = sol_vec_phaseII.loc[sol_vec_phaseII['value']==1]
ref_df = phaseII_model.init_routes_df.set_index('labels')
print(ref_df.loc[['m','lr']][optimal_routes_phaseII.index])
formatted_routes_list_II =  phaseII_model.getRoute4Plot(optimal_routes_phaseII.index.to_list(),
                                                        ref_df,edge_plot_config)
vis_sol.plot_network(formatted_routes_list_II,node_trace,_title="Demo Instance",_cus_dem=customer_demand)

Finish generating variables!
Finish generating constrains!
Finish generating cost vector!....Elapsed-time: 5.505513906478882
Finish generating objective!
        route[99]  route[208]  route[468]  route[1094]
labels                                                
m        1.000000    1.000000    1.000000     2.000000
lr       0.978485    1.426796    1.299101     1.542722
route[99]
route[208]
route[468]
route[1094]


In [828]:
phaseII_model.model.ObjVal

29.81348045979179

### Columns Generations

In [924]:
phaseII_model = md.phaseIIModel(init_route, initializer,
                 distance_matrix,constant_dict)
phaseII_model.buildModel()
phaseII_model.model.setParam('OutputFlag',False)

m_collections = [2,3]
phaseII_model.runColumnsGeneration(m_collections,False,True)

Finish generating variables!
Finish generating constrains!
Finish generating cost vector!....Elapsed-time: 0.28705906867980957
Finish generating objective!
.Running Col. Gen. for m_r: 2 Out-loop-0
..Starting DP subproblem for m' = 2
No. dominated states: 223
Elapsed-time: 28.383567094802856
.Running Col. Gen. for m_r: 3 Out-loop-0
..Starting DP subproblem for m' = 3
No. dominated states: 874
Elapsed-time: 170.37821888923645
.Running Col. Gen. for m_r: 2 Out-loop-1
..Starting DP subproblem for m' = 2
No. dominated states: 223
Elapsed-time: 28.62976837158203
.Running Col. Gen. for m_r: 3 Out-loop-1
..Starting DP subproblem for m' = 3
No. dominated states: 874
Elapsed-time: 171.11018896102905
.Running Col. Gen. for m_r: 2 Out-loop-2
..Starting DP subproblem for m' = 2
No. dominated states: 223
Elapsed-time: 28.67332911491394
.Running Col. Gen. for m_r: 3 Out-loop-2
..Starting DP subproblem for m' = 3
No. dominated states: 874
Elapsed-time: 171.4418637752533
Col.Gen. Completed!...Elapsed-t

In [925]:
timeLimit=120;GAP=0.01;phaseII_model.solveModel()

In [926]:
sol_obj_phaseII = pd.Series(phaseII_model.model.getVars())
sol_vec_phaseII = pd.DataFrame(index = sol_obj_phaseII.apply(lambda x:x.VarName))
sol_vec_phaseII['value'] = sol_obj_phaseII.apply(lambda x:x.X).values
optimal_routes_phaseII = sol_vec_phaseII.loc[sol_vec_phaseII['value']==1]
ref_df = phaseII_model.init_routes_df.set_index('labels')
print(ref_df.loc[['m','lr']][optimal_routes_phaseII.index])
formatted_routes_list_II =  phaseII_model.getRoute4Plot(optimal_routes_phaseII.index.to_list(),
                                                        ref_df,edge_plot_config)
vis_sol.plot_network(formatted_routes_list_II,node_trace,_title="Demo Instance",_cus_dem=customer_demand)

        route[65]  route[99]  route[208]  sDP_C0-864
labels                                              
m        1.000000   1.000000    1.000000           2
lr       1.299101   0.978485    1.426796           0
route[65]
route[99]
route[208]
sDP_C0-864


In [927]:
phaseII_model.model.ObjVal

31.193578032348007

In [928]:
Log = pd.DataFrame(phaseII_model.colgenLogs.values(), index=phaseII_model.colgenLogs.keys())
Log.join(Log.duals.apply(lambda x:x)).drop(columns='duals')

,es_time,cols_gen,cols_add,c_1,c_2,c_3,c_4,c_5,c_6,c_7,c_8,c_9,c_10,m
0,0.000000,0,0,6.624315,3.143744,7.270831,2.862928,4.294852,6.684930,11.135351,4.168207,8.830471,3.131376,-5.125224
1,199.116355,1760,77,5.263465,1.958389,4.629214,0.531187,2.117758,3.960322,5.775744,1.991113,7.566691,2.560041,-1.912272
2,199.772900,1760,1,5.263465,1.958389,4.629214,0.531187,1.579518,3.960322,5.775744,1.938349,7.619455,2.560041,-1.912272
3,200.143085,1760,0,5.263465,1.958389,4.629214,0.531187,1.579518,3.960322,5.775744,1.938349,7.619455,2.560041,-1.912272


In [910]:
# phaseII_model.colgenLogs

In [931]:
phaseII_model = md.phaseIIModel(init_route, initializer,
                 distance_matrix,constant_dict)
phaseII_model.buildModel()
phaseII_model.model.setParam('OutputFlag',False)

m_collections = [2,3]
phaseII_model.runColumnsGeneration(m_collections,False,True,_dominance_rule = 2)

Finish generating variables!
Finish generating constrains!
Finish generating cost vector!....Elapsed-time: 0.29082822799682617
Finish generating objective!
Dominance Checking: True , rule: 2
.Running Col. Gen. for m_r: 2 Out-loop-0
..Starting DP subproblem for m' = 2
No. dominated states: 171
Elapsed-time: 28.131690979003906
.Running Col. Gen. for m_r: 3 Out-loop-0
..Starting DP subproblem for m' = 3
No. dominated states: 741
Elapsed-time: 257.40474104881287
.Running Col. Gen. for m_r: 2 Out-loop-1
..Starting DP subproblem for m' = 2
No. dominated states: 171
Elapsed-time: 28.50071907043457
.Running Col. Gen. for m_r: 3 Out-loop-1
..Starting DP subproblem for m' = 3
No. dominated states: 741
Elapsed-time: 258.84904170036316
.Running Col. Gen. for m_r: 2 Out-loop-2
..Starting DP subproblem for m' = 2
No. dominated states: 171
Elapsed-time: 28.546347856521606
.Running Col. Gen. for m_r: 3 Out-loop-2
..Starting DP subproblem for m' = 3
No. dominated states: 741
Elapsed-time: 257.641542911

In [932]:
timeLimit=120;GAP=0.01;phaseII_model.solveModel()

In [933]:
sol_obj_phaseII = pd.Series(phaseII_model.model.getVars())
sol_vec_phaseII = pd.DataFrame(index = sol_obj_phaseII.apply(lambda x:x.VarName))
sol_vec_phaseII['value'] = sol_obj_phaseII.apply(lambda x:x.X).values
optimal_routes_phaseII = sol_vec_phaseII.loc[sol_vec_phaseII['value']==1]
ref_df = phaseII_model.init_routes_df.set_index('labels')
print(ref_df.loc[['m','lr']][optimal_routes_phaseII.index])
formatted_routes_list_II =  phaseII_model.getRoute4Plot(optimal_routes_phaseII.index.to_list(),
                                                        ref_df,edge_plot_config)
vis_sol.plot_network(formatted_routes_list_II,node_trace,_title="Demo Instance",_cus_dem=customer_demand)

        route[65]  route[99]  route[208]  sDP_C0-864
labels                                              
m        1.000000   1.000000    1.000000           2
lr       1.299101   0.978485    1.426796           0
route[65]
route[99]
route[208]
sDP_C0-864


In [934]:
Log = pd.DataFrame(phaseII_model.colgenLogs.values(), index=phaseII_model.colgenLogs.keys())
Log.join(Log.duals.apply(lambda x:x)).drop(columns='duals')

,es_time,cols_gen,cols_add,c_1,c_2,c_3,c_4,c_5,c_6,c_7,c_8,c_9,c_10,m
0,0.000000,0,0,6.624315,3.143744,7.270831,2.862928,4.294852,6.684930,11.135351,4.168207,8.830471,3.131376,-5.125224
1,286.142302,2436,152,5.242353,2.221626,4.731515,0.531187,2.043882,3.683010,5.085690,1.917237,7.598343,2.436627,-1.891160
2,287.407341,2436,3,5.263465,1.999379,4.752627,0.531187,2.064994,3.919332,5.114294,1.452873,7.619455,2.418580,-1.912272
3,286.232667,2436,0,5.263465,1.999379,4.752627,0.531187,2.064994,3.919332,5.114294,1.452873,7.619455,2.418580,-1.912272


### Columns generation no dominance checking

In [780]:
phaseII_model = md.phaseIIModel(init_route, initializer,
                 distance_matrix,constant_dict)
phaseII_model.buildModel()
phaseII_model.model.setParam('OutputFlag',False)

m_collections = [2,3]
phaseII_model.runColumnsGeneration(m_collections,False,False)

Finish generating variables!
Finish generating constrains!
Finish generating cost vector!....Elapsed-time: 0.28471875190734863
Finish generating objective!
.Running Col. Gen. for m_r: 2 Out-loop-1
..Starting DP subproblem for m' = 2
No. dominated states: 0
Elapsed-time: 31.86065101623535
.Running Col. Gen. for m_r: 3 Out-loop-1
..Starting DP subproblem for m' = 3
No. dominated states: 0
Elapsed-time: 225.91783785820007
.Running Col. Gen. for m_r: 2 Out-loop-2
..Starting DP subproblem for m' = 2
No. dominated states: 0
Elapsed-time: 30.14205574989319
.Running Col. Gen. for m_r: 3 Out-loop-2
..Starting DP subproblem for m' = 3
No. dominated states: 0
Elapsed-time: 220.87370014190674
.Running Col. Gen. for m_r: 2 Out-loop-3
..Starting DP subproblem for m' = 2
No. dominated states: 0
Elapsed-time: 29.76213312149048
.Running Col. Gen. for m_r: 3 Out-loop-3
..Starting DP subproblem for m' = 3
No. dominated states: 0
Elapsed-time: 220.1829001903534
Col.Gen. Completed!...Elapsed-time: 760.2392

In [781]:
timeLimit=120;GAP=0.01;phaseII_model.solveModel()

In [782]:
sol_obj_phaseII = pd.Series(phaseII_model.model.getVars())
sol_vec_phaseII = pd.DataFrame(index = sol_obj_phaseII.apply(lambda x:x.VarName))
sol_vec_phaseII['value'] = sol_obj_phaseII.apply(lambda x:x.X).values
optimal_routes_phaseII = sol_vec_phaseII.loc[sol_vec_phaseII['value']==1]
ref_df = phaseII_model.init_routes_df.set_index('labels')
print(ref_df.loc[['m','lr']][optimal_routes_phaseII.index])
formatted_routes_list_II =  phaseII_model.getRoute4Plot(optimal_routes_phaseII.index.to_list(),
                                                        ref_df,edge_plot_config)
vis_sol.plot_network(formatted_routes_list_II,node_trace,_title="Demo Instance",_cus_dem=customer_demand)

        route[46]  route[100]  sDP_C1-365  sDP_C1-561
labels                                               
m        2.000000    2.000000           3           3
lr       1.117064    0.978485           0           0


In [784]:
# phaseII_model.init_routes_df

In [778]:
# phaseII_model.init_routes_df

In [779]:
m1 = sys.modules['visualize_sol']
m2 = sys.modules['random_instance']
m3 = sys.modules['initialize_path']
m4 = sys.modules['model']
importlib.reload(m1)
importlib.reload(m2)
importlib.reload(m3)
importlib.reload(m4)

<module 'model' from '/Users/ravitpichayavet/Documents/GaTechOR/GraduateResearch/CTC_CVRP/Modules/model.py'>

## (4) DP subproblem
State:= ()

In [507]:
phaseII_model.solveRelaxedModel()
duals_vect = pd.DataFrame(phaseII_model.getDuals(), index = phaseII_model.customers + ['m'])

Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (mac64)
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads
Optimize a model with 6 rows, 17 columns and 39 nonzeros
Model fingerprint: 0x919061a6
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [5e-01, 5e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 5e+00]
Presolve time: 0.01s
Presolved: 6 rows, 17 columns, 39 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   4.500000e+00   0.000000e+00      0s
       9    1.0885890e+01   0.000000e+00   0.000000e+00      0s

Solved in 9 iterations and 0.01 seconds
Optimal objective  1.088589003e+01


In [508]:
phaseII_model.relaxedModel.getConstrs(), phaseII_model.getDuals()

([<gurobi.Constr customer_coverage[3]>,
  <gurobi.Constr customer_coverage[4]>,
  <gurobi.Constr customer_coverage[5]>,
  <gurobi.Constr customer_coverage[6]>,
  <gurobi.Constr customer_coverage[7]>,
  <gurobi.Constr vehicles_usage>],
 [4.799842934667224,
  2.063680826058574,
  2.553093460688674,
  5.3339017452023345,
  0.9352140000158133,
  -0.9599685869334451])

In [33]:
# phaseII_model.relaxedModel.getConstrs(), phaseII_model.getDuals()
# phaseII_model.relaxedModel.getObjective()
# phaseII_model.model.getRow(phaseII_model.relaxedModel.getConstrs()[2])

In [113]:
const_ss = pd.Series(phaseII_model.relaxedModel.getConstrs()).apply(lambda x: x.RHS)
const_ss

0     1.0
1     1.0
2     1.0
3     1.0
4     1.0
5    10.0
dtype: float64

In [17]:
phaseII_model.getRelaxSolution()

0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
dtype: float64

In [19]:
# formatted_routes_list_II =  phaseII_model.getRoute4Plot(['route[0]','route[1]','route[4]'],
#                                                         x,edge_plot_config)
# vis_sol.plot_network(formatted_routes_list_II,node_trace,_title="Demo Instance",_cus_dem=customer_demand)

In [743]:
not(False)

True

In [374]:
m_veh = 2
solution_pricing = pricingDP(m_veh, duals_vect, phaseII_model,np.inf,_print_state=True)
len(solution_pricing['reward'])

Starting DP subproblem for m' = 2
resNode: c_5 deliver_cap: 0.6595264419897522 cap_limit: 10
pi_next 0.0 cost_ij 0.3297632209948761 cost_j0 0.3297632209948761 cost_i0 0
j_customer_demand 1 next_state.accDemand 1 curr_accDistance 0 curr_state.accDemand 0 current_state.accDistance 0
currS 0 O 1.3190528839795044
transReward -0.49464483149231414
nextS 1 c_5 0.8244080524871902
L-sizr: 1
resNode: c_2 deliver_cap: 1.1037122391251288 cap_limit: 10
pi_next 0.44418579713537665 cost_ij 0.5518561195625644 cost_j0 0.5518561195625644 cost_i0 0
j_customer_demand 1 next_state.accDemand 1 curr_accDistance 0 curr_state.accDemand 0 current_state.accDistance 0
currS 0 O 1.3190528839795044
transReward -0.38359838220847
nextS 2 c_2 0.9354545017710344
L-sizr: 2
resNode: c_3 deliver_cap: 1.5931248737552288 cap_limit: 10
pi_next 0.9335984317654766 cost_ij 0.3982812184388072 cost_j0 0.3982812184388072 cost_i0 0
j_customer_demand 2 next_state.accDemand 2 curr_accDistance 0 curr_state.accDemand 0 current_state.ac

j_customer_demand 1 next_state.accDemand 4 curr_accDistance 1.1263256578724905 curr_state.accDemand 3 current_state.accDistance 1.1263256578724905
currS 16 c_5 -0.762303438155649
transReward -1.7660354356153938
nextS 71 c_2 -2.528338873771043
L-sizr: 16
resNode: c_3 deliver_cap: 11.263256578724905 cap_limit: 10
currS 16 c_5 -0.762303438155649
transReward -1.7660354356153938
nextS 73 c_3 Infeasible!
L-sizr: 16
resNode: c_4 deliver_cap: 16.292026479791748 cap_limit: 10
currS 16 c_5 -0.762303438155649
transReward -1.7660354356153938
nextS 74 c_4 Infeasible!
L-sizr: 16
resNode: c_1 deliver_cap: 14.509339997610809 cap_limit: 10
currS 16 c_5 -0.762303438155649
transReward -1.7660354356153938
nextS 75 c_1 Infeasible!
L-sizr: 16
resNode: c_5 deliver_cap: 7.70828802292772 cap_limit: 10
pi_next 0.0 cost_ij 0.24889022829687507 cost_j0 0.3297632209948761 cost_i0 0.5518561195625644
j_customer_demand 1 next_state.accDemand 4 curr_accDistance 1.3484185564401787 curr_state.accDemand 3 current_state.ac

resNode: c_5 deliver_cap: 12.93299223860841 cap_limit: 10
currS 53 c_3 -3.6404189348598206
transReward -1.7256973081120957
nextS 146 c_5 Infeasible!
L-sizr: 5
resNode: c_2 deliver_cap: 15.153921224285295 cap_limit: 10
currS 53 c_3 -3.6404189348598206
transReward -1.7256973081120957
nextS 147 c_2 Infeasible!
L-sizr: 5
resNode: c_4 deliver_cap: 21.725171999509 cap_limit: 10
currS 53 c_3 -3.6404189348598206
transReward -1.7256973081120957
nextS 149 c_4 Infeasible!
L-sizr: 5
resNode: c_1 deliver_cap: 19.258828944794914 cap_limit: 10
currS 53 c_3 -3.6404189348598206
transReward -1.7256973081120957
nextS 150 c_1 Infeasible!
L-sizr: 5
resNode: c_5 deliver_cap: 9.769346677305585 cap_limit: 10
pi_next 0.0 cost_ij 0.24889022829687507 cost_j0 0.3297632209948761 cost_i0 0.5518561195625644
j_customer_demand 1 next_state.accDemand 5 curr_accDistance 1.3752158861693655 curr_state.accDemand 4 current_state.accDistance 1.3752158861693655
currS 71 c_2 -2.528338873771043
transReward -2.1393707780607065
n

37

### Route with reward > 0 

In [375]:
(P,prev,reward) = (solution_pricing['P'],solution_pricing['prev'],solution_pricing['reward'])

In [223]:
print('Nodes being processed:',len(reward))

Nodes being processed: 37


In [224]:
print('Nodes being processed:',len(reward))

Nodes being processed: 37


In [819]:
s1_ss,s0_ss

(3    0
 4    0
 5    0
 6    0
 7    1
 Name: sDP_1, dtype: int64,
 3    0
 4    0
 5    0
 6    0
 7    0
 Name: M, dtype: int64)

In [817]:
(s1_ss>0) == (s0_ss>0)

3     True
4     True
5     True
6     True
7    False
dtype: bool

In [796]:
s0 = reward_ss_plus.index[0]
s1 = reward_ss_plus.index[1]
s1_ss = s1.colDF.loc[s1.colDF.labels.isin(customers)].iloc[:,-1]
s0_ss = s0.colDF.loc[s0.colDF.labels.isin(customers)].iloc[:,-1]

In [729]:
def updateDominance(_cur_s, _unproc_list,_comparing_index):
    cur_cus_visit = _cur_s.colDF.loc[_cur_s.colDF.labels.isin(_comparing_index)].iloc[:,-1]
    cur_cost = _cur_s.route_cost
    dominating_flag = False
    partition_exists = False
    for _state in _unproc_list:
        _s_cus_vist = _state.colDF.loc[_state.colDF.labels.isin(_comparing_index)].iloc[:,-1]
        e_wise_comp = np.all(cur_cus_visit==_s_cus_vist)
        _s_route_cost = _state.route_cost
        print(e_wise_comp)
        if e_wise_comp:
            partition_exists = True
            if cur_cost > _s_route_cost: break
            elif cur_cost < _s_route_cost:
                dominating_flag = True
                _unproc_list.remove(_cur_s)
            elif cur_cost == _s_route_cost:
                dominating_flag = True
        else: partition_exists = partition_exists
    if dominating_flag: _unproc_list.append(_cur_s)
    if not(partition_exists): _unproc_list.append(_cur_s)
    #output flag showing that _cur_s will be added to unproc_list
    return (dominating_flag) or (not(partition_exists))

In [736]:
updateDominance(s1,un_list,customers)

False
True
False
False
False
False
False
True
True
True
True
True


In [737]:
un_list

In [726]:
un_list = reward_ss_plus.index.to_list()
# un_list.remove(s1)

In [377]:
reward_ss = pd.Series(reward)
reward_ss_plus = reward_ss[reward_ss>0.00001]
reward_ss_plus

<model.SPPState object at 0x7f8e8832ce20>    1.319053
<model.SPPState object at 0x7f8e887a2bb0>    0.824408
<model.SPPState object at 0x7f8e888500d0>    0.935455
<model.SPPState object at 0x7f8eedb35cd0>    1.057808
<model.SPPState object at 0x7f8e994b6ac0>    1.797515
<model.SPPState object at 0x7f8e994230a0>    1.619495
<model.SPPState object at 0x7f8e8832fcd0>    0.289567
dtype: float64

In [227]:
for s in reward_ss_plus.index:
    print(md.reconstructPath(s,prev, depot[0]))

['O']
['O', 'c_5']
['O', 'c_2']
['O', 'c_3']
['O', 'c_4']
['O', 'c_1']
['O', 'c_5', 'c_2']


In [228]:
for s in reward_ss_plus.index:
    print(s.colDF)

        labels  M
0           lr  0
1            m  2
2            O  0
3          c_1  0
4          c_2  0
5          c_3  0
6          c_4  0
7          c_5  0
8   (c_5, c_2)  0
9   (c_2, c_5)  0
10    (c_5, O)  0
11    (O, c_5)  0
12  (c_5, c_3)  0
13  (c_3, c_5)  0
14  (c_5, c_4)  0
15  (c_4, c_5)  0
16  (c_5, c_1)  0
17  (c_1, c_5)  0
18    (c_2, O)  0
19    (O, c_2)  0
20  (c_2, c_3)  0
21  (c_3, c_2)  0
22  (c_2, c_4)  0
23  (c_4, c_2)  0
24  (c_2, c_1)  0
25  (c_1, c_2)  0
26    (O, c_3)  0
27    (c_3, O)  0
28    (O, c_4)  0
29    (c_4, O)  0
30    (O, c_1)  0
31    (c_1, O)  0
32  (c_3, c_4)  0
33  (c_4, c_3)  0
34  (c_3, c_1)  0
35  (c_1, c_3)  0
36  (c_4, c_1)  0
37  (c_1, c_4)  0
        labels  sDP_1
0           lr      0
1            m      2
2            O      0
3          c_1      0
4          c_2      0
5          c_3      0
6          c_4      0
7          c_5      1
8   (c_5, c_2)      0
9   (c_2, c_5)      0
10    (c_5, O)      1
11    (O, c_5)      1
12  (c_5, c_

In [350]:
# _n = 'c_4'
# a1 = phaseII_model.distance_matrix[('O','c_5')]/constant_dict['truck_speed']
# a2 = phaseII_model.distance_matrix[('c_5','c_2')]/constant_dict['truck_speed']
# a3 = phaseII_model.distance_matrix[('c_2','O')]/constant_dict['truck_speed']
# _lr = a1+a2+a3
# (_lr*(customer_demand['c_2']+customer_demand['c_5']))/(2*2) + (customer_demand['c_5']*a1+customer_demand['c_2']*(a1+a2))

In [368]:
# Filtering
reward_ss = pd.Series(reward)
reward_ss_plus = reward_ss[reward_ss>0.00001]
for _state in reward_ss_plus.index:
    # Calculate cost route
    duals_idx = pd.Series(duals_vect.index)
    duals_coeff = duals_idx.apply(lambda x: _state.colDF.loc[_state.colDF.labels==x].iloc[:,-1].values[0]*duals_vect.loc[x].values[0])
    route_cost = duals_coeff.sum()-reward_ss_plus[_state]
    _state.route_cost = route_cost
#     print(_state.colDF[_state.colDF.iloc[:,-1]>0])
#     print(duals_coeff)
#     print('cost_route:',duals_coeff.sum()-reward_ss_plus[_state])


In [380]:
[x.route_cost for x in reward_ss_plus.index]
# prepColumns(reward, duals_vect)

[0.0,
 0.4946448314923142,
 0.8277841793438466,
 1.1948436553164217,
 3.4139645713354447,
 2.8799057608003333,
 1.4736714547137852]

In [370]:
# def filteringColumns():
def generateColumns(self,_reward_ss,_duals, ):
    for _state in _reward_ss.index:
        # Calculate cost route
        duals_idx = pd.Series(_duals.index)
        duals_coeff = duals_idx.apply(lambda x: _state.colDF.loc[_state.colDF.labels==x].iloc[:,-1].values[0]*_duals.loc[x].values[0])
        route_cost = duals_coeff.sum()-_reward_ss_plus[_state]
        _state.route_cost = route_cost

        _col = _state.colDF.loc[_state.colDF.index[self.customer_index]].iloc[:,-1].to_list() +_state.colDF.loc[_state.colDF.labels=='m'].iloc[:,-1].to_list()
        newColumn = Column(_col, self.model.getConstrs())
          
        _name = _state.colDF.columns[-1]
        self.addColumn(newColumn,route_cost,_name)

In [460]:
def pricingDP(_m_veh, _duals, _model_instance,_max_stops=None,_print_state=False,_prefix=''):
    '''DP Pricing subproblem
        Input: m_veh:= no. of vehicle using in the route that we're going to search for the best reduced cost
              duals:= vector of dual variables
              _max_stops:= no. of max visited. Default is total number of customers.
        Output: solution_object:=
                better_cols = List of columns having better reduced cost
               node_processed = List of state being processed
               node_fathomed = List of state being dominated
    '''
    if _max_stops is None: _max_stops = len(_model_instance.customers)
    label_counter = 0
    #Master colDF Pattern  
    master_colDF = pd.DataFrame(_model_instance.init_routes_df.loc[:,'labels']);
    _row_label = master_colDF['labels']
    master_colDF['M'] = 0; master_colDF.loc[row_label=='m','M']=_m_veh
    
    #Initialization
    _depot_node = _model_instance.depot[0]
    s_0 = md.SPPState(_depot_node, 0,0,0,0,master_colDF)
    prev = dict(); reward = dict()
    prev[s_0] = -1
    reward[s_0] = _m_veh*_duals.loc['m',0]
    L = [s_0];P = []
    arcs_ss = pd.Series(_model_instance.arcs)
    return_arcs = arcs_ss[arcs_ss.apply(lambda x: x[-1]==depot[0])]            
    return_arcs_cost_dict = pd.Series(index = return_arcs, data = return_arcs.apply(lambda x: distance_matrix[x]/_model_instance.truck_speed).values).to_dict()
    return_arcs_cost_dict[(_depot_node,_depot_node)] = 0; #boundary condition

    
    print('Starting DP subproblem for m\' =', _m_veh)
    t1 = time.time()
    while len(L) != 0:
        currState = L.pop(0)
        if currState.resNode == _depot_node: 
            delta_plus = arcs_ss[arcs_ss.apply(lambda x: x[0]==_depot_node)];
        else: delta_plus = arcs_ss[arcs_ss.apply(lambda x: x[0]==currState.resNode)]
        for arc in delta_plus.to_list():
            label_counter +=1; 
#             if _print_state: print(label_counter,currState.resNode, arc)
            if (arc[-1] == _depot_node) or (currState.nodeVisited == _max_stops): continue
            else:
                ext_node = arc[-1]
                curr_node = arc[0]
                return_cost_ext = return_arcs_cost_dict[(ext_node,_depot_node)]
                return_cost_curr = return_arcs_cost_dict[(curr_node,_depot_node)]
                ext_col = pd.DataFrame(currState.colDF.labels)
                ext_col['sDP_C%s-%s'%(_prefix,label_counter)] = currState.colDF.iloc[:,-1]
                ext_col.loc[currState.colDF.labels==ext_node,'sDP_C%s-%s'%(_prefix,label_counter)]+=1 #Node visited
                ext_col.loc[currState.colDF.labels.isin(return_arcs),'sDP_C%s-%s'%(_prefix,label_counter)] = 0 #Deleted previous return arc
                ext_col.loc[currState.colDF.labels.isin([arc,(ext_node,_depot_node)]),'sDP_C%s-%s'%(_prefix,label_counter)]+=1 #Arc passed
                
            extState = md.SPPState(ext_node, currState.accDemand+customer_demand[ext_node], 
                               currState.accDistance+(_model_instance.distance_matrix[arc]/_model_instance.truck_speed), 
                                currState.nodeVisited+1,label_counter,ext_col)
            
            feas_flag = extState.checkFeasibility(_model_instance.vehicle_capacity,_m_veh,return_cost_ext,_print_status=True)
            if feas_flag:
                prev[extState] = currState
                #Can be improved!
                transition_reward = md.transitionReward(currState,extState,
                                                        _duals,_m_veh,_model_instance,
                                                        return_cost_curr,return_cost_ext)
                reward[extState] = reward[currState]+transition_reward
                L.append(extState)
                
            if _print_state: 
                print('currS',currState.label, currState.resNode,reward[currState])
                print('transReward',transition_reward)
                if feas_flag: print('nextS',extState.label, extState.resNode,reward[extState])
                else: print('nextS',extState.label, extState.resNode,'Infeasible!')
                print('L-sizr:',len(L))
    #             if reward[extState] > 0: break
            else: continue
        P.append(currState)
#         break
#     solution_obj = dict(zip(['better_column','node_processed','node_fathomed'],[]))
    solution_obj = dict(zip(['P','prev','reward'],[P,prev,reward]))
    print('Elapsed-time:',time.time()-t1)
    return solution_obj


In [524]:
a = pd.Series(index = [1,4,3],data=False)
a[4]

False

In [ ]:
def runColumnsGeneration(self,_m_collections):
    self.solveRelaxedModel()
    duals_vect = pd.DataFrame(self.getDuals(), index = self.customers + ['m'])
    opt_cond_vect = pd.Series(index = _m_collections,data=False)
    while opt_cond_vect.sum()<len(_m_collections):
        for _m_veh in _m_collections:
            print('Running Columns Generations for m_r:', _m_veh)
            solution_pricing = pricingDP(_m_veh, duals_vect, self,np.inf,_print_state=True)
            # len(solution_pricing['reward'])
            (P,prev,reward) = (solution_pricing['P'],solution_pricing['prev'],solution_pricing['reward'])
            #Preprocess columns
            reward_ss = pd.Series(reward)
            
            ## Filtering columns
            reward_ss_filtered = reward_ss[reward_ss>0.00001]
            
            if len(reward_ss_filtered)>0:
                #Add columns
                self.generateColumns(reward_ss_filtered, duals_vect)
            else: 
                opt_cond_vect[_m_veh] = True
                continue
        #Resolve relax model
        self.solveRelaxedModel()
        duals_vect = pd.DataFrame(self.getDuals(), index = self.customers + ['m'])
        

In [509]:
##COLUMNS GENERATIONS

#Relaxation

#Pricing
m_veh = 2
solution_pricing = pricingDP(m_veh, duals_vect, phaseII_model,np.inf,_print_state=True)
# len(solution_pricing['reward'])
(P,prev,reward) = (solution_pricing['P'],solution_pricing['prev'],solution_pricing['reward'])

#Preprocess columns
reward_ss = pd.Series(reward)
## Filtering columns
reward_ss_filtered = reward_ss[reward_ss>0.00001]

#Add columns
phaseII_model.generateColumns(reward_ss_filtered, duals_vect)


Starting DP subproblem for m' = 2
resNode: c_5 deliver_cap: 0.6595264419897522 cap_limit: 10
pi_next 0.9352140000158133 cost_ij 0.3297632209948761 cost_j0 0.3297632209948761 cost_i0 0
j_customer_demand 1 next_state.accDemand 1 curr_accDistance 0 curr_state.accDemand 0 current_state.accDistance 0
currS 0 O -1.9199371738668902
transReward 0.4405691685234992
nextS 1 c_5 -1.479368005343391
L-sizr: 1
resNode: c_2 deliver_cap: 1.1037122391251288 cap_limit: 10
pi_next 2.063680826058574 cost_ij 0.5518561195625644 cost_j0 0.5518561195625644 cost_i0 0
j_customer_demand 1 next_state.accDemand 1 curr_accDistance 0 curr_state.accDemand 0 current_state.accDistance 0
currS 0 O -1.9199371738668902
transReward 1.2358966467147274
nextS 2 c_2 -0.6840405271521628
L-sizr: 2
resNode: c_3 deliver_cap: 1.5931248737552288 cap_limit: 10
pi_next 2.553093460688674 cost_ij 0.3982812184388072 cost_j0 0.3982812184388072 cost_i0 0
j_customer_demand 2 next_state.accDemand 2 curr_accDistance 0 curr_state.accDemand 0 cu

resNode: c_1 deliver_cap: 16.346783688591948 cap_limit: 10
currS 13 c_3 -2.284211928812136
transReward -1.8663053352023433
nextS 60 c_1 Infeasible!
L-sizr: 18
resNode: c_5 deliver_cap: 13.956080795565267 cap_limit: 10
currS 14 c_4 -2.3572680877538787
transReward -1.8663053352023433
nextS 61 c_5 Infeasible!
L-sizr: 17
resNode: c_2 deliver_cap: 16.310996429888082 cap_limit: 10
currS 14 c_4 -2.3572680877538787
transReward -1.8663053352023433
nextS 62 c_2 Infeasible!
L-sizr: 17
resNode: c_3 deliver_cap: 17.07316368260551 cap_limit: 10
currS 14 c_4 -2.3572680877538787
transReward -1.8663053352023433
nextS 64 c_3 Infeasible!
L-sizr: 17
resNode: c_1 deliver_cap: 25.688015867711165 cap_limit: 10
currS 14 c_4 -2.3572680877538787
transReward -1.8663053352023433
nextS 65 c_1 Infeasible!
L-sizr: 17
resNode: c_5 deliver_cap: 12.037747366695793 cap_limit: 10
currS 15 c_1 -1.9517191316083577
transReward -1.8663053352023433
nextS 66 c_5 Infeasible!
L-sizr: 16
resNode: c_2 deliver_cap: 14.2586763523726

resNode: c_4 deliver_cap: 17.32523335927595 cap_limit: 10
currS 51 c_2 -0.4607200871014062
transReward -0.7904833080962825
nextS 144 c_4 Infeasible!
L-sizr: 6
resNode: c_1 deliver_cap: 15.381762898719884 cap_limit: 10
currS 51 c_2 -0.4607200871014062
transReward -0.7904833080962825
nextS 145 c_1 Infeasible!
L-sizr: 6
resNode: c_5 deliver_cap: 12.93299223860841 cap_limit: 10
currS 53 c_3 -2.7052049348440077
transReward -0.7904833080962825
nextS 146 c_5 Infeasible!
L-sizr: 5
resNode: c_2 deliver_cap: 15.153921224285295 cap_limit: 10
currS 53 c_3 -2.7052049348440077
transReward -0.7904833080962825
nextS 147 c_2 Infeasible!
L-sizr: 5
resNode: c_4 deliver_cap: 21.725171999509 cap_limit: 10
currS 53 c_3 -2.7052049348440077
transReward -0.7904833080962825
nextS 149 c_4 Infeasible!
L-sizr: 5
resNode: c_1 deliver_cap: 19.258828944794914 cap_limit: 10
currS 53 c_3 -2.7052049348440077
transReward -0.7904833080962825
nextS 150 c_1 Infeasible!
L-sizr: 5
resNode: c_5 deliver_cap: 9.769346677305585 c

In [513]:
# reward_ss[reward_ss>0]

In [510]:
phaseII_model.init_routes_df

,labels,route[0],route[1],route[2],route[3],route[4],route[5],route[6],route[7],route[8],test,M,sDP_C-1,sDP_C-2,sDP_C-3,sDP_C-4,sDP_C-5,sDP_C-6,sDP_C-31
0,lr,1.279958,1.103712,0.796562,1.517318,0.659526,1.13051,1.13051,1.456089,1.456089,0,0,0,0,0,0,0,0,0
1,m,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000,1.000000,1.000000,2,2,2,2,2,2,2,2,2
2,O,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.00000,1.000000,1.000000,0,0,0,0,0,0,0,0,0
3,c_1,1.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0,0,0,0,0,0,1,0,0
4,c_2,0.000000,1.000000,0.000000,0.000000,0.000000,1.00000,1.00000,0.000000,0.000000,1,0,0,1,0,0,0,1,1
5,c_3,0.000000,0.000000,1.000000,0.000000,0.000000,0.00000,0.00000,1.000000,1.000000,0,0,0,0,1,0,0,0,0
6,c_4,0.000000,0.000000,0.000000,1.000000,0.000000,0.00000,0.00000,0.000000,0.000000,0,0,0,0,0,1,0,0,0
7,c_5,0.000000,0.000000,0.000000,0.000000,1.000000,1.00000,1.00000,1.000000,1.000000,1,0,1,0,0,0,0,1,2
8,"(c_5, c_2)",0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,1.00000,0.000000,0.000000,1,0,0,0,0,0,0,1,1
9,"(c_2, c_5)",0.000000,0.000000,0.000000,0.000000,0.000000,1.00000,0.00000,0.000000,0.000000,0,0,0,0,0,0,0,0,1
